In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from gbmcollaboration import AllDataCenters
from gbmcollaboration import (
    SURVIVAL_EXCLUSION
)
from gbmcollaboration.utils import pathology_annotation_loader as patho
from gbmcollaboration.viz.cell2loc_plotting import clustermap
from gbmcollaboration.viz.figure_helpers import savefig, set_mpl_rcparams
from gbmcollaboration.viz import set_mpl_rcparams, savefig
set_mpl_rcparams()
# from gbmcollaboration.utils import get_datasets, s3_ls
# data = get_datasets()

# Params

In [ ]:
path_efs = '/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/'
path_ecm_annot = path_efs+'Analysis_Repo/Notebooks/ecm_neighborhood_annotations_v4.csv'
path_til_annot = path_efs+'results/visium/til_zones/til_zone_annotations.csv'
path_cell_type_info = '../../gbmcollaboration/metadata/Cell type annotation markers Glimmer.xlsx'
FIGURES_DIR = Path('/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/figures/visium/cell2loc_colocalization/')

# Load annotations

In [ ]:
# load ECM and TIL annotations
df_ecm_annot = pd.read_csv(path_ecm_annot,index_col='spot_id')
df_til_annot = pd.read_csv(path_til_annot,index_col='spot_id')

# load pathology annotations
cfg = patho.DataConfig(
    s3_bucket="pa-bi4jtrdsa151p4d3m-jepjxh9qrwgxphe8hd3r6exicahnoeuw1b-s3alias",
    s3_prefixes=(
        "fd319842-9157-4f04-bd4f-7a36ce6a0ba7/Non_mosaic_upload/Mosaic Annotations/",  # CHUV
        "2ca49430-8b5c-4504-ad96-344019c90e84/",  # UKER
    ),
)

acfg = patho.AnalysisConfig(
    min_spots_per_annotation=0,
    keep_only_shared_annotations=False,
)

df_patho_annot = patho.load_and_prepare(cfg, acfg)
df_all_annot = df_ecm_annot.join([df_til_annot[['TIL_positive','TIL_zone']], df_patho['pathology_annotations']])

# Load data

In [ ]:
# data = AllDataCenters.load_spatial(
#     exclude_low_quality_visium = True,
#     exclude_idh1_mut = True,
#     exclude_patient_ids = True,
#     exclude_relapse = True,
#     load_deconvolution=True,
#     load_gex=False
# )

df_q05 = pd.concat([v.celltype_df.drop(columns=['imagerow','imagecol']) for k,v in data.items()]).fillna(0.)
df_q05.index = df_q05.index.to_flat_index().str.join('_')

# map cell types names to paper names and create simplified deconv matrix
df_markers, df_ct_map = pd.read_excel(path_cell_type_info,sheet_name=None).values()
assert all(df_q05.columns.isin(_map.index))

# assert index matches with metadata
assert all(df_q05.index == df_all_annot.index)

# normalize
df_q05_norm = df_q05.div(df_q05.sum(axis=1), axis=0)

# join with metadata for each level and create simplified deconv matrix
# also get average per annotation
annot_keys = [
    'TIL_zone',
    'ECM_neighborhood',
    'pathology_annotations'
]

dfs_q05_norm = {}
df_q05_norm_mean_per_annot = {}
for i in tqdm(range(1,5)):
    lvl = f'celltype_level{i}_curated'
    _map = df_ct_map.set_index('celltype_level4_scanvi')[lvl]
    
    dfs_q05_norm[lvl] = df_q05_norm.T.groupby(_map).sum().T.join(df_all_annot)
    ct_cols = dfs_q05_norm[lvl].columns.difference(df_all_annot.columns)

    df_q05_norm_mean_per_annot[lvl] = {}
    for k in annot_keys:
        df_q05_norm_mean_per_annot[lvl][k] = dfs_q05_norm[lvl].groupby(k)[ct_cols].mean(0).T

# Plots

In [ ]:
plot_annots = {
    'TIL_zone': None,
    'ECM_neighborhood': None,
    'pathology_annotations': None
}

for i in tqdm(range(1,5)):
    lvl = f'celltype_level{i}_curated'

    for k, cat_subset in plot_annots.items():

        if k == 'ECM_neighborhood':
            col_prefix = 'ECM_'
        elif k == 'TIL_zone':
            col_prefix = ''

        df_q05_norm_mean = df_q05_norm_mean_per_annot[lvl][k]

        upper = np.quantile(df_q05_norm_mean,0.99)
        df_plot = df_q05_norm_mean.copy()
        df_plot = df_plot.clip(upper=upper)
        df_plot.columns = col_prefix+df_plot.columns.astype(str)

        if cat_subset is not None:
            df_plot = df_plot.loc[cat_subset]

        clustermap(df_plot, figure_size=(14, 14), 
                    cmap='RdPu', log=True, fun_type='dotplot',
                    cluster_rows=True, cluster_cols=True)


        # save plot
        out_path = Path(FIGURES_DIR) / f'dotplot_{lvl}_{k}.png'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        savefig(df_plot, out_path, close=True)